These notebook shows what experiments are ran in this study to investigate the effectiveness of the LLM-based GP approach. 

The experiments require a Daytona API key and Together API key to be set up and initialised as environment variables (as TOGETHER_AI and DAYTONA_API_KEY, respectively.)

To deploy an experiment as a Docker container, copy the relevant experiments into the main portion of the experiments.py file, and build the container using the provided Dockerfile.

Caution: Running all these experiments can take days, particularly when multiple LLM models are compared.

## Setup

In [ ]:
#Setup DEAP
import os
from deap import base, creator, tools, gp
from experiments import tune_gp_model
from experiments import model_comparisons
from experiments import blackbox_vs_groundtruth
from experiments import compare_two_approaches
from experiments import create_daytona_snapshot

#Create results folder
try:
    os.mkdir("results")
except FileExistsError:
    pass

#Only run one - creates snapshot remotely on Daytona account with DEAP installed
create_daytona_snapshot()

#Set up creator object
creator.create("FitnessMin", base.Fitness, weights=(-1.0,)) #We want to minimise fitness
creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMin) #Individuals are GP trees (with an associated fitness value)


# Experiments

1. Tune baseline GP (no LLM) - tune_gp_model
2. Tune k (plot k) - tune_gp_model
3. Tune self-adapt + default temp (no plot) - tune_gp_model
4. Compare multiple models - model_comparisons(params, names)
5. Black box vs Ground Truth for optimal setup - blackbox_vs_groundtruth(optimal_parameters, standard_model_params, model_name)
6. Compare optimal LLM-GP against Standard GP - compare_two_approaches(dataset, alg1_name, alg2_name, alg1_params, alg2_params, num_runs=10)
7. Compare optimal LLM-GP against GP with Fixed Operator Design - compare_two_approaches(dataset, alg1_name, alg2_name, alg1_params, alg2_params, num_runs=10)
8. Compare Fixed Operator Design GP against Standard GP - compare_two_approaches(dataset, alg1_name, alg2_name, alg1_params, alg2_params, num_runs=10)



## Hyperparmater Tuning

### 1. Baseline Model

In [ ]:
tuning_ranges = {
    "pop_size": [100, 300, 500], #250
    "cxpb": [0.5, 0.7, 0.9],
    "mutpb": [0.01, 0.1, 0.2],
    "tourn_size": [3, 5, 7],
    "k": [1000],
    "self_adapt_req": [None],
    "default_temperature": [0.3],
    "temperature_alpha": [0.1],
    "model": [None],
    "reasoning_model": [False]
}

tune_gp_model(tuning_ranges, "results/standard_tuning", plot_param=None)


### 2. Tuning K

In [ ]:
k_ranges = {
    "pop_size": [300],
    "cxpb": [0.9],
    "mutpb": [0.05],
    "tourn_size": [7],
    "k": [2, 3, 4, 5, 6, 7],
    "self_adapt_req": [None],
    "default_temperature": [0.3],
    "temperature_alpha": [0.1],
    "model": ["openai/gpt-oss-120b"],
    "reasoning_model": [True]
}

tune_gp_model(k_ranges, "results/k_tuning", plot_param="k")

### 3. Tuning Self-Adaptation Rate

In [ ]:
self_adaptation_ranges = {
    "pop_size": [300], 
    "cxpb": [0.9],
    "mutpb": [0.05],
    "tourn_size": [7],
    "k": [5],
    "self_adapt_req": [5, 7, 9], 
    "default_temperature": [0.3, 0.5, 0.7],
    "temperature_alpha": [0.05, 0.1, 0.2],
    "model": ["openai/gpt-oss-120b"],
    "reasoning_model": [True]
}

disabled_self_adaptation = {
    "pop_size": [300], 
    "cxpb": [0.9],
    "mutpb": [0.05],
    "tourn_size": [7],
    "k": [5],
    "self_adapt_req": [None], 
    "default_temperature": [0.2, 0.4, 0.6, 0.8, 1.0],
    "temperature_alpha": [0.1],
    "model": ["openai/gpt-oss-120b"],
    "reasoning_model": [True]
}

tune_gp_model(self_adaptation_ranges, "results/self_adaption_params", plot_param=None)
tune_gp_model(disabled_self_adaptation, "results/default_temp", plot_param="default_temperature")


## 4. Model Comparison

In [ ]:
models = ["meta-llama/Llama-3.3-70B-Instruct-Turbo", "deepseek-ai/DeepSeek-V3.1", "Qwen/Qwen3-Coder-Next-FP8", "openai/gpt-oss-120b", "moonshotai/Kimi-K2.5", None]
model_names = ["Llama 3.3 70b", "DeepSeek V3.1", "Qwen3 Coder", "GPT OSS 120b", "Kimi K2.5", "No LLM"]
reasoning = [False, True, False, True, True, False]

optimal_parameters = {
    "pop_size": 300, 
    "gens": 100,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.9,
    "mutpb": 0.05,
    "tourn_size": 7,
    "k": 5, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "ln"],
    "verbose": True,
    "self_adapt_req": 5,
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 100,
    "model": None,
    "reasoning_model": False
}

#Gets parameters for each model
parameters = []
for i in range(len(models)):
    current_parameters = optimal_parameters.copy()
    current_parameters["model"] = models[i]
    current_parameters["reasoning_model"] = reasoning[i]
    parameters.append(current_parameters)

model_comparisons(parameters, model_names)


## 5 + 6. Statistical Testing + Black-box vs Ground Truth Problems

In [ ]:
optimal_parameters = {
    "pop_size": 300, 
    "gens": 100,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.9,
    "mutpb": 0.05,
    "tourn_size": 7,
    "k": 5, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "ln"],
    "verbose": True,
    "self_adapt_req": 5,
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 20,
    "model": "openai/gpt-oss-120b",
    "reasoning_model": True
}

standard_params = {
    "pop_size": 300, 
    "gens": 100,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.9,
    "mutpb": 0.05,
    "tourn_size": 7,
    "k": 5, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "ln"],
    "verbose": True,
    "self_adapt_req": 5,
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 20,
    "model": None,
    "reasoning_model": False
}

blackbox_vs_groundtruth(optimal_parameters, standard_params, "GPT-OSS-120b")

## 7. LLM-GP vs Optimal Design

In [ ]:
#Insert filepath to optimal LLM-generated designs
mutation_design_filepath = "docs/default_mutation_design.txt"
crossover_desing_filepath = "docs/default_crossover_design.txt"

fixed_design_params = {
    "pop_size": 300, 
    "gens": 100,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.9,
    "mutpb": 0.05,
    "tourn_size": 7,
    "k": 5, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "ln"],
    "verbose": True,
    "self_adapt_req": 5,
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 20,
    "model": None,
    "reasoning_model": False,
    "custom_crossover_filepath": crossover_desing_filepath,
    "custom_mutation_filepath": mutation_design_filepath
}

blackbox_vs_groundtruth(optimal_parameters, fixed_design_params, "GPT-OSS-120b")

## 8. Optimal Design vs Standard GP

In [ ]:
#Insert filepath to optimal LLM-generated designs
mutation_design_filepath = "docs/default_mutation_design.txt"
crossover_desing_filepath = "docs/default_crossover_design.txt"

fixed_design_params = {
    "pop_size": 300, 
    "gens": 100,
    "max_time": 8.0 * 60.0 * 60.0,
    "cxpb": 0.9,
    "mutpb": 0.05,
    "tourn_size": 7,
    "k": 5, 
    "functions": ["+", "-", "*", "/", "sqrt", "sin", "cos", "ln"],
    "verbose": True,
    "self_adapt_req": 5,
    "default_temperature": 0.3,
    "temperature_alpha": 0.1,
    "maximum_stagnation": 20,
    "model": None,
    "reasoning_model": False,
    "custom_crossover_filepath": crossover_desing_filepath,
    "custom_mutation_filepath": mutation_design_filepath
}

blackbox_vs_groundtruth(fixed_design_params, standard_params, "GPT-OSS-120b")